# BRISC 2025 Baseline — Attention Residual U-Net

**Target:** Tumour Dice ≥ 0.80 (BRISC is harder than CAMUS because tumours are small ~1.7% of image and vary widely in shape).

## Kaggle setup
1. Add datasets: `briscdataset/brisc2025`  +  `iteris-pkg`
2. Accelerator: GPU T4 x2
3. Persistence: Files only
4. Save Version → Save & Run All (Commit)

**Path auto-detection** lives in Cell 1 — if your BRISC dataset is mounted somewhere unusual, edit `BRISC_ROOT_OVERRIDE` to a hardcoded path.

> ### Phase guide -- which notebook/data for which phase
> This notebook -- which trains the attention-U-Net competitor (BRISC) -- is the **Phase A** (full-dataset) version. Run it as-is, no changes needed.
>
> | Phase | Data | Notebook |
> |---|---|---|
> | **A** (full data) | full dataset | **this notebook**, unmodified |
> | **B** (low-data regime) | ~155 training images | `notebooks/phaseB/unet/04_brisc_attnunet.ipynb` |
> | **C** (MSA backbone, smaller subset) | ~93 training images | not yet available -- needs the archived MSA backbone wired into `AGENT_REGISTRY` first (see `docs/EXPERIMENTS.md` section 4) |
>
> Full methodology, exact sizes, and why: **`docs/EXPERIMENTS.md`**.


## 0 · Install (no kernel restart needed)

In [ ]:
import subprocess
from pathlib import Path

# Locate requirements.txt + iteris package — handles Kaggle's wrapper-folder quirk
init_files = list(Path('/kaggle/input').rglob('iteris/__init__.py'))
if not init_files:
    raise RuntimeError('iteris-pkg not attached.')
PKG_ROOT = init_files[0].parent.parent
REQ      = PKG_ROOT / 'requirements.txt'

subprocess.run(['pip', 'install', '-r', str(REQ), '--quiet', '--upgrade'], check=True)
print(f'✓ Setup complete. Package at: {PKG_ROOT}')

## 1 · Load config + locate the BRISC dataset

In [ ]:
import sys
from pathlib import Path

# Locate the iteris package (handles any Kaggle wrapper layout)
init_files = list(Path('/kaggle/input').rglob('iteris/__init__.py'))
if not init_files:
    raise RuntimeError('iteris-pkg not attached.')
PKG_ROOT = init_files[0].parent.parent
sys.path.insert(0, str(PKG_ROOT))

from iteris.config import load_config
from iteris.utils  import get_device

cfg = load_config(str(PKG_ROOT / 'configs' / 'BRISC' / 'brisc.yaml'))

# Locate BRISC's segmentation_task folder (handles any Kaggle path nesting)
seg_dirs = [p for p in Path('/kaggle/input').rglob('segmentation_task') if p.is_dir()]
if not seg_dirs:
    raise RuntimeError('BRISC segmentation_task folder not found. Attach briscdataset/brisc2025.')
cfg['data_root']      = str(seg_dirs[0])
cfg['checkpoint_dir'] = '/kaggle/working'

print(f'Package root : {PKG_ROOT}')
print(f'Data root    : {cfg["data_root"]}')
print(f'Dataset      : {cfg["dataset"]}  ({cfg["modality"]})')
print(f'Image size   : {cfg["image_size"]}  in_channels={cfg["in_channels"]}')
print(f'Classes      : {cfg["num_classes"]} — {cfg["class_names"]}')
print(f'Normalisation: {cfg["normalize"]}   binarise labels: {cfg["binarize_labels"]}')
print(f'Epochs       : {cfg["epochs"]}  batch {cfg["batch_size"]}  lr {cfg["lr"]}')
print(f'Device       : {get_device()}')

## 2 · Train

In [ ]:
from iteris.training import run_training

result      = run_training(cfg)
model       = result['model']
history     = result['history']
best_dice   = result['best_dice']
test_loader = result['test_loader']
checkpoint  = result['checkpoint']

print(f'\n✓ Training done. Best val Dice = {best_dice:.4f}  |  Checkpoint: {checkpoint}')

## 3 · Learning curves

In [ ]:
from iteris.visualization import plot_learning_curves
plot_learning_curves(history, cfg, target_dice=0.80)

## 4 · Test-set evaluation

In [ ]:
from iteris.evaluation import evaluate_test_set
scores_df = evaluate_test_set(model, test_loader, cfg)
print(scores_df.head())

## 5 · Export predicted masks (init state for DRL agents)

In [ ]:
from iteris.evaluation import export_predicted_masks
export_predicted_masks(model, test_loader, cfg)

## 6 · Qualitative grid

In [ ]:
from iteris.visualization import plot_qualitative_grid
plot_qualitative_grid(model, test_loader, cfg, n_show=4)

## 7 · Summary JSON

In [ ]:
from iteris.evaluation import save_summary_json
save_summary_json(history, scores_df, cfg, best_dice)